# AgriVision: Crop Row Analizi - Eksiksiz Veri Çıktısı

Bu versiyon, görev dokümanındaki tüm tablo gereksinimlerini (Koordinat, Uzunluk, Alan) karşılamaktadır.

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

GSD_CM_PER_PX = 0.5
PX2_PER_M2 = 40000

def imread_unicode(path, flags=cv2.IMREAD_COLOR):
    try: return cv2.imdecode(np.fromfile(path, dtype=np.uint8), flags)
    except: return None

def imwrite_unicode(path, img):
    try:
        _, ext = os.path.splitext(path)
        result, nparray = cv2.imencode(ext, img)
        if result: nparray.tofile(path)
    except: pass

BASE_DIR = os.getcwd()
TEST_IMG_DIR = os.path.join(BASE_DIR, 'test_data', 'image')
TEST_LBL_DIR = os.path.join(BASE_DIR, 'test_data', 'label')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

def segment_rows_optimized(label_mask):
    if label_mask is None: return None, []
    _, binary = cv2.threshold(label_mask, 50, 255, cv2.THRESH_BINARY)
    h, w = binary.shape
    binary_cut = binary.copy()
    binary_cut[0:int(h*0.25), :] = 0
    num_labels, labels_map, stats, centroids = cv2.connectedComponentsWithStats(binary_cut, connectivity=4)
    rows = []
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] > 100:
            rows.append({'id': i, 'centroid': centroids[i], 'bbox': stats[i]})
    rows.sort(key=lambda x: x['centroid'][0])
    return labels_map, rows

def detect_gaps(image, labels_map, rows, min_gap_area=500):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    plant_mask = cv2.inRange(hsv, lower_green, upper_green)
    plant_mask = cv2.morphologyEx(plant_mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    all_gaps = []
    for idx, row in enumerate(rows):
        row_mask = (labels_map == row['id']).astype(np.uint8) * 255
        row_mask_dilated = cv2.dilate(row_mask, np.ones((25, 25), np.uint8))
        gap_candidate = cv2.bitwise_and(row_mask_dilated, cv2.bitwise_not(plant_mask))
        gap_candidate = cv2.morphologyEx(gap_candidate, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
        num_gaps, gap_labels, g_stats, g_centroids = cv2.connectedComponentsWithStats(gap_candidate)
        for g_i in range(1, num_gaps):
            area = g_stats[g_i, cv2.CC_STAT_AREA]
            if area > min_gap_area:
                all_gaps.append({
                    'row_idx': idx + 1, 
                    'bbox': g_stats[g_i], 
                    'centroid': g_centroids[g_i], 
                    'area_px': area
                })
    return all_gaps

In [ ]:
test_files = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.jpg')])
gap_data = []

for filename in test_files:
    img_path = os.path.join(TEST_IMG_DIR, filename)
    lbl_path = os.path.join(TEST_LBL_DIR, filename)
    image = imread_unicode(img_path)
    label = imread_unicode(lbl_path, cv2.IMREAD_GRAYSCALE)
    if image is None or label is None: continue
    
    l_map, rows = segment_rows_optimized(label)
    gaps = detect_gaps(image, l_map, rows)
    
    for g in gaps:
        x, y, w, h, _ = g['bbox']
        gap_data.append({
            'gap_id': f"Gap-{len(gap_data)+1:03d}",
            'row_id': f"Row-{g['row_idx']}",
            'image_filename': filename,
            'x_center_px': round(g['centroid'][0], 1),
            'y_center_px': round(g['centroid'][1], 1),
            'width_px': w,
            'height_px': h,
            'area_px2': g['area_px'],
            'area_m2': round(g['area_px'] / PX2_PER_M2, 6)
        })

df = pd.DataFrame(gap_data)
df.to_csv(os.path.join(OUTPUT_DIR, 'gap_analysis_final.csv'), index=False)
print(f"Eksiksiz CSV kaydedildi: {len(df)} kayıt.")
print(df.head())
